# Entrenamiento de modelos de calidad de agua

Este notebook documenta la etapa de entrenamiento de modelos de aprendizaje supervisado para la estimación de parámetros de calidad de agua en humedales.

A partir del split generado en la etapa de preparación del dataset, se entrenan tres algoritmos de regresión:

- Support Vector Regression (SVR),
- Gradient Boosting Regressor (GBR),
- Random Forest Regressor (RFR).

Cada modelo se entrena mediante búsqueda de hiperparámetros con `GridSearchCV`, utilizando el mismo conjunto de entrenamiento para garantizar consistencia en la comparación posterior.

## Objetivo

Entrenar y almacenar tres modelos de regresión para estimar un parámetro de calidad de agua a partir de variables espectrales derivadas de imágenes multibanda.

El flujo incluye:

1. carga del split de entrenamiento y prueba,
2. recuperación de metadatos del modelado,
3. entrenamiento de SVR, GBR y RFR,
4. búsqueda de hiperparámetros con validación cruzada,
5. almacenamiento de los modelos entrenados,
6. almacenamiento de los resultados de búsqueda,
7. almacenamiento de los mejores hiperparámetros encontrados.

In [1]:
from pathlib import Path
import sys

BASE_DIR = Path.cwd()

while not (BASE_DIR / "src").exists() and BASE_DIR != BASE_DIR.parent:
    BASE_DIR = BASE_DIR.parent

sys.path.append(str(BASE_DIR))

print("BASE_DIR detectado:")
print(BASE_DIR)

BASE_DIR detectado:
C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales


## Importación de funciones auxiliares

Se importan funciones desde los módulos ubicados en `src/calidad_agua/`.

En este notebook se utiliza:

- `cargar_split()` para recuperar el conjunto de entrenamiento y prueba generado previamente,
- `pipeline_entrenamiento_modelos()` para entrenar y guardar los tres modelos.

In [2]:
from src.calidad_agua.preparacion import cargar_split
from src.calidad_agua.entrenamiento import pipeline_entrenamiento_modelos

## Definición de rutas de entrada y salida

Se define la ruta del archivo `.joblib` que contiene el split de entrenamiento y prueba. Este archivo fue generado en el notebook:

`02_preparacion_dataset_modelos.ipynb`

También se definen las carpetas donde se almacenarán:

- los modelos entrenados,
- los objetos de búsqueda de hiperparámetros,
- y los mejores hiperparámetros encontrados.

In [3]:
# ============================================================
# PARÁMETRO DE SALIDA
# ============================================================

# Debe coincidir con el nombre limpio usado en el notebook 02.
# Ejemplo: si target_modelo = "ln_DQO", entonces parametro_salida = "DQO".
parametro_salida = "DQO"

ruta_split = BASE_DIR / "data" / "interim" / "calidad_agua" / "splits" / f"split_{parametro_salida}.joblib"

print("Ruta del split:")
print(ruta_split)

Ruta del split:
C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales\data\interim\calidad_agua\splits\split_DQO.joblib


## Carga del split y metadatos del modelado

Se carga el archivo `.joblib` con el split de entrenamiento y prueba.

Este archivo contiene:

- `X_train`,
- `X_test`,
- `y_train`,
- `y_test`,
- `target`,
- `vars_pred`,
- `aplicar_log`,
- `columnas_log`,
- `test_size`,
- `random_state`.

Aunque este notebook sólo utiliza directamente `X_train` y `y_train` para entrenar los modelos, los demás elementos se conservan como metadatos para garantizar trazabilidad del flujo.

In [4]:
split_data = cargar_split(ruta_split)

X_train = split_data["X_train"]
X_test = split_data["X_test"]
y_train = split_data["y_train"]
y_test = split_data["y_test"]

target_modelo = split_data["target_modelo"]
target_salida = split_data["target_salida"]
vars_pred = split_data["vars_pred"]

crear_ln = split_data["crear_ln"]
aplicar_log = split_data["aplicar_log"]
param_cols_log = split_data["param_cols_log"]
valor_reemplazo_log = split_data["valor_reemplazo_log"]
n_registros = split_data["n_registros"]

test_size = split_data["test_size"]
random_state = split_data["random_state"]

print("Split cargado correctamente.")

print("\nVariable objetivo usada para modelado:")
print(target_modelo)

print("\nNombre limpio para salidas:")
print(target_salida)

print("\n¿Se crearon columnas logarítmicas?")
print(crear_ln)

print("\n¿El target usa transformación logarítmica?")
print(aplicar_log)

print("\nColumnas base transformadas:")
print(param_cols_log)

print("\nValor de reemplazo logarítmico:")
print(valor_reemplazo_log)

print("\nNúmero de registros usado:")
print(n_registros)

print("\nDimensiones:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Split cargado correctamente.

Variable objetivo usada para modelado:
ln_DQO

Nombre limpio para salidas:
DQO

¿Se crearon columnas logarítmicas?
True

¿El target usa transformación logarítmica?
True

Columnas base transformadas:
['DQO', 'Fosfatos', 'Turbidez', 'Nitratos', 'Sulfatos', 'ficocianin', 'Chl', 'CE']

Valor de reemplazo logarítmico:
0.3

Número de registros usado:
65

Dimensiones:
X_train: (55, 17)
X_test: (10, 17)
y_train: (55,)
y_test: (10,)


## Definición de rutas de salida

A partir de `target_salida` se definen las rutas donde se almacenarán los modelos, resultados de búsqueda y mejores hiperparámetros.

Esto permite que, aunque el modelo se entrene con una variable transformada como `ln_DQO`, las carpetas y archivos de salida se organicen bajo el nombre limpio del parámetro, por ejemplo `DQO`.

In [5]:
carpeta_modelos = BASE_DIR / "models" / "calidad_agua" / "entrenados" / target_salida

ruta_busquedas = BASE_DIR / "models" / "calidad_agua" / "metricas" / target_salida / "gridsearch_modelos.joblib"

ruta_hiperparametros = BASE_DIR / "outputs" / "tables" / "calidad_agua" / target_salida / "mejores_hiperparametros.joblib"

print("Carpeta de modelos:")
print(carpeta_modelos)

print("\nRuta de búsquedas GridSearchCV:")
print(ruta_busquedas)

print("\nRuta de mejores hiperparámetros:")
print(ruta_hiperparametros)

Carpeta de modelos:
C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales\models\calidad_agua\entrenados\DQO

Ruta de búsquedas GridSearchCV:
C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales\models\calidad_agua\metricas\DQO\gridsearch_modelos.joblib

Ruta de mejores hiperparámetros:
C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales\outputs\tables\calidad_agua\DQO\mejores_hiperparametros.joblib


## Configuración del entrenamiento

En esta sección se definen los parámetros generales de entrenamiento.

Se usa `GridSearchCV` para cada modelo, con validación cruzada. La métrica de optimización definida en el módulo de entrenamiento es `neg_root_mean_squared_error`, de modo que la búsqueda prioriza configuraciones con menor RMSE.

El parámetro `n_jobs=-1` permite usar todos los núcleos disponibles del equipo durante la búsqueda.

In [6]:
cv = 5
scoring = "neg_root_mean_squared_error"
n_jobs = -1
verbose = 2

print("Configuración de entrenamiento:")
print("cv:", cv)
print("scoring:", scoring)
print("n_jobs:", n_jobs)
print("verbose:", verbose)

Configuración de entrenamiento:
cv: 5
scoring: neg_root_mean_squared_error
n_jobs: -1
verbose: 2


## Entrenamiento de modelos

La siguiente celda ejecuta el entrenamiento de los tres modelos:

- SVR,
- Gradient Boosting Regressor,
- Random Forest Regressor.

Cada modelo se ajusta mediante `GridSearchCV`, utilizando su propio espacio de hiperparámetros. Al finalizar, se guardan los modelos entrenados en la carpeta correspondiente al parámetro de calidad de agua modelado.

In [7]:
modelos_entrenados, resultados_busqueda, rutas_modelos = pipeline_entrenamiento_modelos(
    X_train=X_train,
    y_train=y_train,
    carpeta_modelos=carpeta_modelos,
    ruta_busquedas=ruta_busquedas,
    ruta_hiperparametros=ruta_hiperparametros,
    random_state=random_state,
    cv=cv,
    scoring=scoring,
    n_jobs=n_jobs,
    verbose=verbose
)


Entrenando modelo SVR
Fitting 5 folds for each of 64 candidates, totalling 320 fits

Entrenando modelo GBR
Fitting 5 folds for each of 1728 candidates, totalling 8640 fits

Entrenando modelo RFR
Fitting 5 folds for each of 1800 candidates, totalling 9000 fits

✅ Entrenamiento de modelos finalizado.
Modelo guardado en: C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales\models\calidad_agua\entrenados\DQO\SVR.joblib
Modelo guardado en: C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales\models\calidad_agua\entrenados\DQO\GBR.joblib
Modelo guardado en: C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales\models\calidad_agua\entrenados\DQO\RFR.joblib
Resultados de búsqueda guardados en: C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales\models\calidad_agua\metricas\DQO\gridsearch_modelos.joblib
Mejores hiperparámetros guardados en: C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales\outputs\tables\calidad_agua\DQO\mejores_hiperparametros.joblib


## Modelos almacenados

Se revisan las rutas donde fueron guardados los modelos entrenados.

La estructura esperada es:

`models/calidad_agua/entrenados/<PARAMETRO>/`

Por ejemplo, para `DQO`:

- `SVR.joblib`,
- `GBR.joblib`,
- `RFR.joblib`.

In [8]:
print("Rutas de modelos guardados:")

for nombre_modelo, ruta_modelo in rutas_modelos.items():
    print(f"{nombre_modelo}: {ruta_modelo}")

Rutas de modelos guardados:
SVR: C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales\models\calidad_agua\entrenados\DQO\SVR.joblib
GBR: C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales\models\calidad_agua\entrenados\DQO\GBR.joblib
RFR: C:\Users\AVDON\JupyterLab\ATENEA\Proyecto_Atenea_Humedales\models\calidad_agua\entrenados\DQO\RFR.joblib


## Mejores hiperparámetros encontrados

Se revisan los mejores hiperparámetros identificados por `GridSearchCV` para cada modelo.

Estos resultados permiten documentar la configuración final seleccionada para SVR, Gradient Boosting Regressor y Random Forest Regressor.

In [9]:
for nombre_modelo, search in resultados_busqueda.items():
    print("\n==============================")
    print(f"Modelo: {nombre_modelo}")
    print("==============================")
    print("Mejores hiperparámetros:")
    print(search.best_params_)
    print("\nMejor puntuación de validación cruzada:")
    print(search.best_score_)


Modelo: SVR
Mejores hiperparámetros:
{'model__C': 0.1, 'model__epsilon': 0.01, 'model__gamma': 'scale', 'model__kernel': 'rbf'}

Mejor puntuación de validación cruzada:
-0.5810336344557265

Modelo: GBR
Mejores hiperparámetros:
{'learning_rate': 0.01, 'max_depth': 2, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 100, 'subsample': 0.7}

Mejor puntuación de validación cruzada:
-0.6892694757426463

Modelo: RFR
Mejores hiperparámetros:
{'bootstrap': True, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 8, 'n_estimators': 500}

Mejor puntuación de validación cruzada:
-0.6996942527971509


## Productos generados

Al finalizar este notebook se generan los siguientes productos:

1. Modelos entrenados en formato `.joblib`:
   - `SVR.joblib`,
   - `GBR.joblib`,
   - `RFR.joblib`.

2. Archivo con los objetos completos de búsqueda `GridSearchCV`.

3. Archivo con los mejores hiperparámetros encontrados para cada modelo.

Estos productos permiten reproducir, evaluar e interpretar los modelos entrenados en las siguientes etapas del flujo.

## Consideraciones técnicas

- Los tres modelos se entrenan con el mismo split para permitir una comparación justa.
- SVR se implementa dentro de un `Pipeline` con `StandardScaler`, debido a su sensibilidad a la escala de las variables.
- Gradient Boosting Regressor y Random Forest Regressor no requieren escalamiento previo bajo esta configuración.
- La variable objetivo usada para entrenamiento se almacena como `target_modelo`.
- Si `target_modelo` corresponde a una columna transformada, como `ln_DQO`, los modelos se entrenan en escala logarítmica natural.
- Las carpetas y archivos de salida se organizan usando `target_salida`, por ejemplo `DQO`, para facilitar la interpretación de los productos.
- Si se cambia el parámetro objetivo, las variables predictoras, la transformación logarítmica o el número de registros, debe regenerarse el split y reentrenarse el conjunto de modelos.

## Siguiente etapa del flujo

Una vez entrenados y almacenados los modelos, el siguiente paso consiste en evaluar su desempeño y comparar sus métricas sobre los mismos conjuntos de entrenamiento y prueba.

Esta fase se documenta en:

`04_evaluacion_modelos.ipynb`